## Requisitos

Antes de ejecutar el cuaderno, instala las dependencias desde la raíz del proyecto:

```bash
pip install -r requirements.txt
```

Las credenciales se leen de un fichero `.env` local (ver `.env.example`). Cada autor debe ejecutar el cuaderno con su propia cuenta de Spotify y su propia `LASTFM_API_KEY`.

Este código solo da los top artistas, sin género (no requiere de LastFM):

In [ ]:
# import pandas as pd
   
# RANGOS = {
#     "short_term": "ultimas_4_semanas",
#     "medium_term": "ultimos_6_meses",
#     "long_term": "varios_anios",
# }
 
# filas = []
# for rango, etiqueta in RANGOS.items():
#     resultado = sp.current_user_top_artists(limit=50, time_range=rango)
#     for posicion, artista in enumerate(resultado["items"], start=1):
#         filas.append(
#             {
#                 "rango": etiqueta,
#                 "posicion": posicion,
#                 "artista": artista["name"],
#                 "generos": "; ".join(artista.get("genres", [])),
#                 "spotify_url": artista["external_urls"]["spotify"],
#             }
#         )
 
# df = pd.DataFrame(filas)
# df.to_csv("top_artistas.csv", index=False, encoding="utf-8-sig")
 
# # Vista rápida: top 10 de cada rango
# for etiqueta in RANGOS.values():
#     print(f"\n=== {etiqueta} ===")
#     top = df[df["rango"] == etiqueta].head(10)
#     for _, fila in top.iterrows():
#         print(f"{fila['posicion']:>2}. {fila['artista']}")
 
# print("\nGuardado en top_artistas.csv")

Este da el top artistas con 4 géneros (se pueden pedir más géneros, requiere de LastFM):

In [ ]:
"""
Guarda tus artistas más escuchados de Spotify, con géneros de Last.fm,
en un CSV con las columnas:
    rango,posicion,artista,generos,spotify_url

Cada autor ejecuta el cuaderno con su propia cuenta. Cambia USUARIO antes
de ejecutar para no sobrescribir el fichero del otro autor.

Requisitos:
    pip install -r requirements.txt

Fichero .env en la misma carpeta (ver .env.example):
    SPOTIPY_CLIENT_ID=tu_client_id
    SPOTIPY_CLIENT_SECRET=tu_client_secret
    SPOTIPY_REDIRECT_URI=http://127.0.0.1:8888/callback
    LASTFM_API_KEY=tu_api_key
"""

import os
import time

import pandas as pd
import requests
import spotipy
from dotenv import load_dotenv
from spotipy.oauth2 import SpotifyOAuth

load_dotenv()

USUARIO = "pablo"  # "pablo" | "jorge"
ARCHIVO_SALIDA = {
    "pablo": "top_artistas.csv",
    "jorge": "top_artistasJorge.csv",
}[USUARIO]

LASTFM_KEY = os.getenv("LASTFM_API_KEY")
if not LASTFM_KEY:
    raise SystemExit("Falta LASTFM_API_KEY en el fichero .env")

RANGOS = {
    "short_term": "ultimas_4_semanas",
    "medium_term": "ultimos_6_meses",
    "long_term": "varios_anios",
}

MIN_PESO = 20     # peso mínimo (0-100) de una etiqueta de Last.fm
MAX_GENEROS = 4   # géneros guardados por artista

# Etiquetas de Last.fm que no son géneros musicales
IGNORAR = {
    "seen live", "female vocalists", "male vocalists", "american", "british",
    "spanish", "french", "german", "swedish", "canadian", "australian",
    "singer-songwriter", "favorites", "favourite", "awesome", "legend",
}


def generos_de(artista: str) -> str:
    params = {
        "method": "artist.gettoptags",
        "artist": artista,
        "api_key": LASTFM_KEY,
        "format": "json",
        "autocorrect": 1,
    }
    try:
        r = requests.get("https://ws.audioscrobbler.com/2.0/", params=params, timeout=10)
        r.raise_for_status()
        etiquetas = r.json().get("toptags", {}).get("tag", [])
    except (requests.RequestException, ValueError):
        return ""

    generos = []
    for e in etiquetas:
        nombre = e["name"].strip().lower()
        if int(e.get("count", 0)) >= MIN_PESO and nombre not in IGNORAR:
            generos.append(nombre)
        if len(generos) == MAX_GENEROS:
            break
    return "; ".join(generos)


sp = spotipy.Spotify(auth_manager=SpotifyOAuth(scope="user-top-read"))

filas = []
for rango, etiqueta in RANGOS.items():
    resultado = sp.current_user_top_artists(limit=50, time_range=rango)
    for posicion, artista in enumerate(resultado["items"], start=1):
        filas.append(
            {
                "rango": etiqueta,
                "posicion": posicion,
                "artista": artista["name"],
                "spotify_url": artista["external_urls"]["spotify"],
            }
        )

df = pd.DataFrame(filas)

# Una sola consulta a Last.fm por artista, aunque salga en varios rangos
cache = {}
for nombre in df["artista"].unique():
    cache[nombre] = generos_de(nombre)
    print(f"{nombre}: {cache[nombre] or '(sin géneros)'}")
    time.sleep(0.25)  # respeta el límite de peticiones

df["generos"] = df["artista"].map(cache)

df = df[["rango", "posicion", "artista", "generos", "spotify_url"]]
df.to_csv(ARCHIVO_SALIDA, index=False, encoding="utf-8-sig")
print(f"\nGuardado en {ARCHIVO_SALIDA}")
